# Fine-Tune OpenLID via PyTorch Head Expansion
This notebook implements a novel incremental learning methodology for FastText models. 
Since FastText `.bin` models cannot dynamically expand their internal vocabulary/label dictionaries out-of-the-box, we use the pre-trained OpenLID model as a frozen feature extractor and migrate its classification head to PyTorch. This allows us to mathematically expand the output matrix to support new languages without catastrophic forgetting (when combined with a replay buffer).

In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing base dependencies...")
    os.system('pip install -q pandas numpy torch huggingface_hub')
    # fasttext wheel can sometimes be tricky on colab, so we compile it
    os.system('pip install -q fasttext')
    print("Setup complete!")


## Step 1: Download and Setup the Base OpenLID Model
We use the `HPLT/OpenLID-v3` model from Hugging Face. This model is compact and perfectly suited for this expansion architecture because it was trained using standard softmax, making its output matrix mathematically sound for 1:1 migration.

In [2]:
import os
from huggingface_hub import hf_hub_download
import fasttext

# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

model_dir = 'models/pretrained/openlid'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'openlid-v3.bin')

if not os.path.exists(model_path):
    print("Downloading OpenLID-v3 from HuggingFace...")
    downloaded_path = hf_hub_download(repo_id="HPLT/OpenLID-v3", filename="openlid-v3.bin")
    # Symlink or move it to our local models dir
    os.system(f'cp {downloaded_path} {model_path}')
    print(f"Model downloaded and saved to {model_path}")
else:
    print(f"Model already exists at {model_path}")

print("Loading FastText model (this may take a moment)...")
# Suppress fasttext warning
fasttext.FastText.eprint = lambda x: None
ft_model = fasttext.load_model(model_path)
print("OpenLID model loaded successfully!")

old_labels = ft_model.get_labels()
hidden_dim = ft_model.get_dimension()
print(f"Original Model supports {len(old_labels)} languages.")
print(f"Hidden Dimension: {hidden_dim}")


/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model downloaded and saved to models/pretrained/openlid/openlid-v3.bin
Loading FastText model (this may take a moment)...
OpenLID model loaded successfully!
Original Model supports 195 languages.
Hidden Dimension: 256
